# 🛠️ Notebook 2: Airline Management — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/airline-management
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from itertools import count

class SeatClass(Enum):
    ECONOMY = "ECON"; BUSINESS = "BIZ"; FIRST = "FIRST"

PRICING = {SeatClass.ECONOMY: 200, SeatClass.BUSINESS: 700, SeatClass.FIRST: 1800}

@dataclass(frozen=True)
class Passenger:
    id: int
    name: str
    passport: str

@dataclass(frozen=True)
class Seat:
    number: str             # e.g. "12A"
    seat_class: SeatClass

@dataclass
class Aircraft:
    model: str
    seats: list[Seat]


In [ ]:
_bids = count(1)

@dataclass
class Flight:
    number: str
    origin: str
    destination: str
    departs: datetime
    aircraft: Aircraft
    # availability: seat_number -> is_available
    _available: dict[str, bool] = field(init=False)

    def __post_init__(self):
        self._available = {s.number: True for s in self.aircraft.seats}

    def available_seats(self, seat_class=None) -> list[Seat]:
        return [s for s in self.aircraft.seats
                if self._available[s.number]
                and (seat_class is None or s.seat_class == seat_class)]

    def book(self, passenger: Passenger, seat: Seat) -> "Booking":
        if not self._available.get(seat.number, False):
            raise ValueError(f"seat {seat.number} not available")
        self._available[seat.number] = False
        return Booking(passenger=passenger, flight=self, seat=seat,
                       price=PRICING[seat.seat_class])

@dataclass
class Booking:
    passenger: Passenger
    flight: Flight
    seat: Seat
    price: float
    id: int = field(default_factory=lambda: next(_bids))

    def ticket(self) -> str:
        return (f"TKT#{self.id}  {self.flight.number}  {self.flight.origin}→"
                f"{self.flight.destination}  seat {self.seat.number} "
                f"({self.seat.seat_class.value})  ${self.price}")


In [ ]:
# 4 economy + 2 business + 1 first
seats = ([Seat(f"{r}{c}", SeatClass.ECONOMY) for r in (5,6) for c in "AB"]
         + [Seat(f"{r}{c}", SeatClass.BUSINESS) for r in (2,)   for c in "AB"]
         + [Seat("1A", SeatClass.FIRST)])
plane = Aircraft("A320", seats)

fl = Flight("UA100","SFO","JFK",
            datetime(2025,6,1,9,0), plane)

alice = Passenger(1,"Alice","P111")
bob   = Passenger(2,"Bob","P222")

print("Available business seats:", fl.available_seats(SeatClass.BUSINESS))

b1 = fl.book(alice, fl.available_seats(SeatClass.BUSINESS)[0])
b2 = fl.book(bob,   fl.available_seats(SeatClass.ECONOMY)[0])
print(b1.ticket()); print(b2.ticket())

# Double-book fails
try: fl.book(bob, b1.seat)
except ValueError as e: print("expected:", e)


### Try it
- Add `cancel(booking)` that frees the seat and refunds based on lead time.
- Add a `Crew` (pilot, flight attendant) assigned to a Flight.
- Add a `Search` service across many flights: `search(origin, dest, date)`.